# 02: DeepEval, a RAG chatbot tested component by component and end to end

**System under test:** the member-help chatbot of Harbourlight Health, a fictional
health insurer. Members ask questions about their plan; the chatbot answers from 10
short policy documents. All data is synthetic.

**Question this notebook answers:** when a chatbot gives a wrong answer, can testing
each component separately show *where* it went wrong, which an end-to-end score alone
cannot?

| Section | What it does | LLM cost |
|---|---|---|
| 0 | How the chatbot works, and how it is tested | None |
| 1 | Set-up, and a check that the expected answers are themselves correct | None |
| 2 | Test the retriever alone | Judge only |
| 3 | Test the generator alone, given the correct information | Generator + judge |
| 4 | Test end to end, baseline k=3 (Option A), repeated 3 times | Generator + judge |
| 5 | Fault injection: break the retriever and see which scores move | Generator + judge |
| 6 | Judge calibration: compare the LLM judge with human labels | Judge only |
| 7 | Fix and retest: k=5 (Option B) against the baseline | Generator + judge |
| 8 | What I learned | None |

**Summary of results:** *to be written once all sections have run.*

## 0. How the chatbot works, and how it is tested

The chatbot is a **RAG** (retrieval-augmented generation) system. Instead of
answering from what the language model happens to know, it first looks up the
relevant parts of the policy documents and then asks the model to answer from those
alone. It has two components, and each can fail on its own:

1. **Retriever** (`rag_pipeline.py`, scikit-learn TF-IDF): finds the pieces of the
   documents most likely to contain the answer. If it finds the wrong pieces, the
   answer will be wrong however good the model is.
2. **Generator** (`gpt-6-luna`): writes the answer from the pieces it is given. If it
   misreads them, or adds things they do not say, the answer is wrong even when
   retrieval was perfect.

The diagram shows the test set-up: each component of the chatbot (left, purple) has
its own evaluation (right, green). A deliberate fault (amber) is injected into the
retriever later, to see which evaluations notice.

![Day 3 evaluation set-up](images/day3_eval_setup.png)

**Step 1: the documents are cut into chunks.** The chatbot never reads a whole
document; it is handed a few small pieces ("chunks"). How the documents are cut
matters. The healthy setting cuts at paragraph breaks and keeps the page title on
every piece. The faulty setting cuts blindly every 200 characters, which can split a
word ("F" / "itness") or a rule across two pieces.

![How documents are split into chunks](images/day3_chunking.png)

**Step 2: the retriever ranks every chunk against the question.** TF-IDF scores each
chunk from 0 to 1 by how many important words it shares with the question. Rare words
such as "acupuncture" count for more than common ones such as "plan". Only the top
**k** chunks are passed to the generator (k=3 in the baseline). In this real example
the chunk holding the answer ranks 4th, just below the cut-off, so the generator never
sees it. This is the kind of failure component testing is designed to expose.

![How the retriever ranks chunks and applies k](images/day3_retrieval.png)

**Step 3: the generator answers from those k chunks only.**

## 1. Set-up

**What this section shows:** the environment is working, and the test data is sound
before any model is tested.

- **Test basis** (what the system should do): the 10 member documents in
  `datasets/policy_docs/`.
- **Oracle** (how we decide right from wrong): the 15-question golden set in
  `datasets/rag_golden.csv`. Each question has an expected answer, the documents it
  should come from, and a few **key facts** the answer depends on.
- **Judge:** `gpt-6-sol`, a stronger model that scores the LLM-based metrics.

DeepEval telemetry is switched off (`DEEPEVAL_TELEMETRY_OPT_OUT=YES` in `.env`) and no
Confident AI cloud login is used, so every result stays in this repo.

### Step 1.1: environment
Loads the API key from `.env`, prints the package versions (so results can be
reproduced), and creates the `results/` folders.

In [ ]:
# Step 1.1: environment. Load .env BEFORE importing deepeval so the telemetry
# opt-out is already set when DeepEval starts.
import os, sys, subprocess
from importlib.metadata import version
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()
os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")   # fallback if .env lacks it

import pandas as pd
import matplotlib.pyplot as plt

RESULTS = Path("results"); (RESULTS / "charts").mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 120)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY missing from .env"
print("Python", sys.version.split()[0])
for pkg in ("deepeval", "scikit-learn", "openai", "pandas"):
    print(f"{pkg:<13}", version(pkg))
print("Telemetry opt-out:", os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"])

### Step 1.2: check the oracle against the test basis
A test is only as good as its expected results. Before testing the chatbot, this step
checks that every key fact in the golden set really appears in the documents it is
supposed to come from, and that the "unanswerable" questions really are not answered
anywhere. If this fails, any later score would be meaningless.

In [ ]:
# Step 1.2: run datasets/validate_golden.py.
# Run as a subprocess: the folder name 'datasets' clashes with the Hugging Face
# 'datasets' package, so it is not imported as a module.
check = subprocess.run([sys.executable, "datasets/validate_golden.py"],
                       capture_output=True, text=True)
print(check.stdout)
assert check.returncode == 0, "Golden set does not match the policy documents"

**Reading the output:** `PASS: 15/15` means every expected answer is supported by the
documents. Two documents are never an expected source (the plan overview and the
contact page). That is deliberate: they mention many topics, so they act as realistic
distractors for the retriever.

### Step 1.3: build the system under test
Loads the documents, cuts them into chunks in both modes (healthy and faulty), and
loads the golden set.

In [ ]:
# Step 1.3: the system under test and the golden set.
from rag_pipeline import build_retriever, load_docs, DEFAULT_K
from rag_metrics import load_golden, golden_context, retriever_report, is_answerable

golden = load_golden()
retrievers = {mode: build_retriever(mode) for mode in ("paragraph", "fixed")}

print(f"{len(load_docs())} documents")
for mode, r in retrievers.items():
    print(f"chunk_mode={mode:<9} -> {len(r.chunks)} chunks")
pd.DataFrame(golden)["question_type"].value_counts().rename("questions")

**Reading the output:** paragraph mode gives 49 chunks and fixed mode 51, so the two
settings hand the retriever a similar number of pieces; what differs is *where* the
cuts fall. The golden set has 8 standard questions, 3 unanswerable ones (the bot should
say it does not know), 2 adversarial ones (a prompt injection and a false premise) and
2 refusals (medical advice, a promise of payment).

## 2. Test the retriever alone

**What this section shows:** how often the retriever hands the generator the
information it needs, measured *before* any answer is written. If the information
never arrives, no generator can answer correctly, so a wrong end-to-end answer can be
traced back here. This is the test pyramid applied to AI: test the component before
the system.

Only the 12 answerable questions are used. The 3 unanswerable questions have no correct
source to find.

| Term | Meaning |
|---|---|
| Chunk | A piece of a document. The generator sees only the chunks it is given. |
| `chunk_mode` | `paragraph` (healthy): cut at blank lines, page title kept. `fixed` (fault): cut every 200 characters. |
| `k` | How many top-scoring chunks are passed to the generator. Baseline k=3. |
| Doc rank | Position of the first chunk from a correct document in the ranking (1 = top). |
| Doc hit@k | Is any chunk from a correct document in the top k? Yes or no. |
| Key-fact recall@k | Share of the question's key facts found in the text of the top k chunks (0 to 1). |
| All facts present | Did the generator receive every fact the answer needs? |

### 2.1 Deterministic metrics (no LLM, free, repeatable)
These checks are plain text matching, so they give the same result every run and cost
nothing. They come first because a cheap, exact check is better than an LLM judge
wherever it is enough.

### Step 2.1: summary for each configuration
Runs the retriever on every answerable question in four configurations (two chunk
modes, each at k=1 and k=3) and totals the results. Per-question results are saved to
`results/retriever_deterministic.csv`.

In [ ]:
# Step 2.1: deterministic retriever metrics for each chunk mode at k=1 and k=3.
det = pd.DataFrame([row | {"chunk_mode": mode}
                    for mode, r in retrievers.items()
                    for k in (1, 3)
                    for row in retriever_report(r, golden, k)])
det.to_csv(RESULTS / "retriever_deterministic.csv", index=False)

n_facts = {g["question_id"]: len(g["key_facts"]) for g in golden}
det["facts_found"] = det["key_fact_recall"] * det["question_id"].map(n_facts)
total_facts = sum(n_facts[q] for q in det.question_id.unique())

summary = (det.groupby(["chunk_mode", "k"])
              .agg(questions=("question_id", "count"),
                   doc_hit=("doc_hit", "sum"),
                   facts_found=("facts_found", "sum"),
                   mean_key_fact_recall=("key_fact_recall", "mean"),
                   all_facts_present=("all_facts_present", "sum"))
              .reset_index())
summary["facts_found"] = summary["facts_found"].round().astype(int).astype(str) + f"/{total_facts}"
summary["mean_key_fact_recall"] = summary["mean_key_fact_recall"].round(2)
summary

**Reading the output:** one row per configuration; the baseline is `paragraph`, `k=3`.

| Column | Meaning |
|---|---|
| `doc_hit` | Questions (out of 12) where a chunk from a correct document made the top k |
| `facts_found` | Key facts found in the retrieved text, out of all 27 key facts |
| `mean_key_fact_recall` | The same idea averaged per question, so every question counts equally |
| `all_facts_present` | Questions where *every* needed fact arrived: the most a perfect generator could get fully right |

**What it shows** (the numbers are deterministic, so they should match on every run):
- Even the healthy baseline delivers every needed fact for only 8 of 12 questions. The
  other 4 cannot be answered fully correctly however good the generator is.
- **k=1 is the big fault**: recall falls from 0.74 to 0.46, because one chunk is
  rarely enough.
- **The fixed split is a subtle fault**: at k=3 recall only falls from 0.74 to 0.72,
  although `all_facts_present` drops from 8 to 6. A single average hides that.
- **`doc_hit` flatters the retriever.** At k=1 it rates the *faulty* fixed mode higher
  than the healthy one (8 against 6). It counts the right document even when the
  wrong piece of it was returned, which is why key-fact recall is the main measure here.

### Step 2.2: per-question view of the baseline
Shows each question in the baseline configuration, so individual failures can be
traced. `retrieved` lists the three chunks passed to the generator, as
`document#chunk`.

In [ ]:
# Step 2.2: per-question view of the baseline (paragraph, k=3).
base = det.query("chunk_mode == 'paragraph' and k == @DEFAULT_K")
base[["question_id", "question_type", "doc_rank", "doc_hit",
      "key_fact_recall", "retrieved"]].round(2)

**Reading the output:** look for rows where `key_fact_recall` is below 1.
- **Q02 (acupuncture):** `doc_hit` is True but recall is 0. The right document was
  found, but the wrong paragraph of it (the mixed-receipt rule, not the exclusions
  list). This is the example in the retrieval diagram in section 0.
- **Q03 (joined three months ago, physiotherapy):** the waiting-period document ranks
  12th. The question shares no important words with "182 days" or "waiting period", so
  a word-matching retriever cannot connect them. A person would see the link at once.
- **Q04 (accident in the waiting period):** only 1 of 3 facts arrives; the answer needs
  pieces from three different documents.
- **Q14 (medical advice):** the first chunk from the cover-and-limits document ranks 4th, just outside k=3.

These are the questions where the end-to-end run (section 4) is expected to struggle,
and the reason will be retrieval, not the generator.

### Step 2.3: chart
Draws the key-fact recall for the four configurations and saves it to
`results/charts/retriever_key_fact_recall.png`.

In [ ]:
# Step 2.3: chart. One bar per configuration; key-fact recall is the main measure
# because doc hit@k counts the right document even when the wrong chunk is returned.
order = [("paragraph", 3, "Paragraph, k=3\n(baseline)"), ("fixed", 3, "Fixed, k=3"),
         ("paragraph", 1, "Paragraph, k=1"), ("fixed", 1, "Fixed, k=1")]
vals = [summary.query("chunk_mode == @m and k == @k")["mean_key_fact_recall"].item()
        for m, k, _ in order]

INK, INK2, SERIES, SURFACE = "#0b0b0b", "#52514e", "#2a78d6", "#fcfcfb"
fig, ax = plt.subplots(figsize=(7, 3.6), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
bars = ax.bar([lbl for *_, lbl in order], vals, width=0.5, color=SERIES)
ax.bar_label(bars, labels=[f"{v:.0%}" for v in vals], padding=3, color=INK, fontsize=10)
ax.set_ylim(0, 1); ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_ylabel("Mean key-fact recall", color=INK2)
ax.set_title("Retriever alone: share of needed facts reaching the generator",
             color=INK, fontsize=11, loc="left")
ax.grid(axis="y", color="#e4e3df", linewidth=0.6); ax.set_axisbelow(True)
for side in ("top", "right", "left"): ax.spines[side].set_visible(False)
ax.tick_params(colors=INK2, length=0)
fig.tight_layout()
fig.savefig(RESULTS / "charts" / "retriever_key_fact_recall.png", dpi=150)
plt.show()

**Reading the output:** each bar is the average share of needed facts that reached the
generator. Halving the information (k=1) costs far more than cutting the documents
badly. This chart is the "before" picture for the fault-injection section, which will
show whether the end-to-end scores move in the same way.

### 2.2 DeepEval contextual metrics (LLM judge: `gpt-6-sol`)

**What this adds:** the deterministic checks above only look for exact phrases. An LLM
judge reads the retrieved text for meaning, so it can credit a chunk that says the same
thing in different words, and it explains its score in a sentence. The cost is that the
judge can itself be wrong, which is why it is compared with the deterministic results
here and calibrated against human labels in section 6.

| Metric | Question the judge answers |
|---|---|
| Contextual precision | Are the useful chunks ranked above the useless ones? |
| Contextual recall | Does the retrieved text contain what the expected answer needs? |
| Contextual relevancy | How much of the retrieved text is relevant to the question? |

Each metric returns a **score** from 0 to 1, a **pass or fail** (DeepEval's default pass
mark is 0.5; our own threshold is set after calibration), and a **reason** written by
the judge. No answer is generated in this section, so only the judge is called.

### Step 2.4: load DeepEval and the judge

In [ ]:
# Step 2.4: DeepEval imports and the judge model.
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (ContextualPrecisionMetric, ContextualRecallMetric,
                              ContextualRelevancyMetric)
from deepeval.models import OpenAIModel   # GPTModel is deprecated in 4.x

JUDGE_MODEL = "gpt-6-sol"
judge = OpenAIModel(model=JUDGE_MODEL)    # reads OPENAI_API_KEY from the environment
print("Judge:", judge.get_model_name())

### Step 2.5: helper functions
Three small functions used for the rest of the notebook:

| Helper | What it does |
|---|---|
| `retriever_case()` | Builds one DeepEval **test case** for a question: the question, the expected answer from the golden set, and the chunks the retriever returned. A test case is DeepEval's unit of testing, much like one row of test data with its expected result. |
| `run_metrics()` | Runs each chosen metric on each test case and collects the results in a table: question, metric, score, pass/fail and the judge's reason. It creates a fresh metric for every case, because DeepEval metrics keep the last result internally. |
| `show_reasons()` | Prints the judge's reasons in full, one per row. Tables cut long text short, and the reason is what shows *why* a score was given. |

In [ ]:
# Step 2.5: helpers to build retriever test cases, run metrics and read reasons.
def retriever_case(g, retriever, k=DEFAULT_K):
    """One test case: question, expected answer and the top-k retrieved chunks.
    No actual_output: retriever metrics judge the chunks, not an answer
    (the field is optional in DeepEval 4.x)."""
    chunks, _ = retriever.retrieve(g["question"], k)
    return LLMTestCase(input=g["question"],
                       expected_output=g["expected_answer"],
                       retrieval_context=[c.text for c in chunks])

def run_metrics(cases, metric_classes, label):
    """Score each (question_id, test case) with a FRESH metric instance per case.
    Returns one row per case x metric: score, pass/fail at DeepEval's default
    threshold, and the judge's reason."""
    rows = []
    for qid, case in cases:
        for cls in metric_classes:
            m = cls(model=judge, include_reason=True, async_mode=False)
            m.measure(case)
            rows.append({"run": label, "question_id": qid, "metric": cls.__name__,
                         "score": m.score, "passed": m.is_successful(),
                         "reason": m.reason})
    return pd.DataFrame(rows)

def show_reasons(df):
    """Print every judge reason in full (tables truncate long text)."""
    for r in df.itertuples():
        print(f"{r.question_id}  {r.metric}  score={r.score:.2f}  passed={r.passed}")
        print(f"   {r.reason}\n")

### Step 2.6: smoke test
One metric (contextual recall) on three questions, which is three judge calls. This
confirms the API key, the judge model and the test-case fields work before spending on
the full run. Results are saved to `results/deepeval_smoke.csv`.

In [ ]:
# Step 2.6: SMOKE TEST: 1 metric x 3 questions (3 judge evaluations).
smoke_ids = ["Q01", "Q02", "Q05"]
by_id = {g["question_id"]: g for g in golden}
smoke = run_metrics([(q, retriever_case(by_id[q], retrievers["paragraph"])) for q in smoke_ids],
                    [ContextualRecallMetric], label="smoke")
smoke.to_csv(RESULTS / "deepeval_smoke.csv", index=False)
show_reasons(smoke)

**Reading the output:** for each question, check three things.
1. A score was returned, so the set-up works.
2. The score agrees with the deterministic result where you would expect it to. Q01 and
   Q05 had all key facts retrieved, so a high recall is expected. Q02 had none (the
   acupuncture list was not retrieved), so a low recall is expected.
3. The reason refers to what is actually in the retrieved chunks. A reason that
   mentions facts not in the chunks would be a sign the judge is not reading carefully.

### Step 2.7: full retriever run
**Checkpoint:** if the smoke test returned scores and sensible reasons, set
`RUN_FULL_RETRIEVER = True` and run the cell. That is 12 questions x 3 metrics = 36
judge evaluations, on the baseline configuration only. Results are saved to
`results/deepeval_retriever_paragraph_k3.csv`, including every reason in full.

In [ ]:
# Step 2.7: full DeepEval retriever run, baseline only (paragraph, k=3).
RUN_FULL_RETRIEVER = False

if RUN_FULL_RETRIEVER:
    cases = [(g["question_id"], retriever_case(g, retrievers["paragraph"]))
             for g in golden if is_answerable(g)]
    ctx = run_metrics(cases, [ContextualPrecisionMetric, ContextualRecallMetric,
                              ContextualRelevancyMetric], label="retriever_paragraph_k3")
    ctx.to_csv(RESULTS / "deepeval_retriever_paragraph_k3.csv", index=False)

    # Side by side: judge scores next to the deterministic key-fact recall.
    wide = ctx.pivot(index="question_id", columns="metric", values="score").round(2)
    wide = wide.join(base.set_index("question_id")[["key_fact_recall", "doc_rank"]].round(2))
    display(wide)
else:
    print("Skipped: set RUN_FULL_RETRIEVER = True after a successful smoke test.")

**Reading the output:** one row per question, with the three judge scores next to the
deterministic key-fact recall and document rank. The interesting rows are where
`ContextualRecallMetric` and `key_fact_recall` disagree. For each of those, read the
reason with `show_reasons(ctx.query("question_id == 'Q..'"))`. Either the key facts are
too narrow (an oracle problem to fix in the golden set) or the judge is too lenient or
too harsh (a judge problem, carried into section 6).

**Observations:** *to be written after the run.*